# Cont. on Applications of PostGIS Functions
In this notebook, we will look at applications of our PostGIS data and functions in machine learning

In [1]:
import psycopg2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import warnings
warnings.filterwarnings("ignore")

First, let's look back at our OSM and GADM datasets:

In [2]:
conn = psycopg2.connect(dbname="postgis", 
                 user="gsa2022", 
                 password="g5!V%T1Vmd", 
                 host="192.168.212.99", 
                 port=32771)

In [3]:
# Check contents of ph_point
df = pd.read_sql('''
SELECT *
FROM ph_point
LIMIT 10
''',conn)
df

,osm_id,access,addr:housename,addr:housenumber,addr:interpolation,admin_level,aerialway,aeroway,amenity,area,...,tourism,tower:type,tunnel,water,waterway,wetland,width,wood,z_order,way
0,4332833465,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,0101000020E6100000EBA7FFAC790C5C4030963325A3B0...
1,5127953795,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,0101000020E610000087527B116D105C4055FC3ACB87BB...
2,4332833991,None,None,None,None,None,None,None,ferry_terminal,None,...,None,None,None,None,None,None,None,None,None,0101000020E610000022CC481861105C40F863A428E1BB...
3,2645876983,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,0101000020E61000007E1EA33C73105C40505A136635BC...
4,2645876976,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,0101000020E6100000060BCCAF8B105C40EFE192E34EBD...
5,4332856711,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,0101000020E6100000A206787789165C402614C7269EDC...
6,2645876984,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,0101000020E610000078D2C26595165C40B222B42FEADC...
7,2074532777,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,0101000020E610000010429B7777175C401789BFDCDDDD...
8,3649052507,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,0101000020E61000009BBD7D6745FB5D406D1915DD1FE4...
9,2120425972,None,None,None,None,None,None,None,ferry_terminal,None,...,None,None,None,None,None,None,None,None,None,0101000020E6100000613024CC59A75D40C364AA6054EA...


In [4]:
# Check the contents of gadm.ph
df2 = pd.read_sql('''
SELECT *
FROM gadm.ph
LIMIT 10
''',conn)
df2

,gid,gid_0,name_0,gid_1,name_1,nl_name_1,gid_2,name_2,varname_2,nl_name_2,type_2,engtype_2,cc_2,hasc_2,geom
0,1,PHL,Philippines,PHL.1_1,Abra,None,PHL.1.1_1,Bangued,None,None,Bayan|Munisipyo,Municipality,140101,PH.AB.BN,0106000020E610000001000000010300000001000000C7...
1,2,PHL,Philippines,PHL.1_1,Abra,None,PHL.1.2_1,Boliney,None,None,Bayan|Munisipyo,Municipality,140102,PH.AB.BL,0106000020E6100000010000000103000000010000001F...
2,3,PHL,Philippines,PHL.1_1,Abra,None,PHL.1.3_1,Bucay,None,None,Bayan|Munisipyo,Municipality,140103,PH.AB.BU,0106000020E61000000100000001030000000100000072...
3,4,PHL,Philippines,PHL.1_1,Abra,None,PHL.1.4_1,Bucloc,None,None,Bayan|Munisipyo,Municipality,140104,PH.AB.BC,0106000020E6100000010000000103000000010000000F...
4,5,PHL,Philippines,PHL.1_1,Abra,None,PHL.1.5_1,Daguioman,None,None,Bayan|Munisipyo,Municipality,140105,PH.AB.DG,0106000020E6100000010000000103000000010000001C...
5,15,PHL,Philippines,PHL.1_1,Abra,None,PHL.1.15_1,Malibcong,None,None,Bayan|Munisipyo,Municipality,140115,PH.AB.ML,0106000020E61000000100000001030000000100000026...
6,1514,PHL,Philippines,PHL.75_1,Surigao del Sur,None,PHL.75.1_1,Barobo,None,None,Bayan|Munisipyo,Municipality,166801,PH.SS.BR,0106000020E61000000300000001030000000100000014...
7,6,PHL,Philippines,PHL.1_1,Abra,None,PHL.1.6_1,Danglas,None,None,Bayan|Munisipyo,Municipality,140106,PH.AB.DN,0106000020E61000000100000001030000000100000070...
8,7,PHL,Philippines,PHL.1_1,Abra,None,PHL.1.7_1,Dolores,None,None,Bayan|Munisipyo,Municipality,140107,PH.AB.DL,0106000020E61000000100000001030000000100000090...
9,8,PHL,Philippines,PHL.1_1,Abra,None,PHL.1.8_1,La Paz,None,None,Bayan|Munisipyo,Municipality,140108,PH.AB.LP,0106000020E610000001000000010300000001000000A1...


In [5]:
#Let's filter out the amenities. Consider only top 20 amenities
df3 = pd.read_sql('''
SELECT amenity, count(amenity) as count
FROM ph_point
GROUP BY amenity
ORDER BY count DESC
LIMIT 20
''', conn)
df3

,amenity,count
0,restaurant,8869
1,school,5853
2,bank,5369
3,fast_food,5249
4,place_of_worship,4652
5,fuel,3799
6,pharmacy,2956
7,cafe,2171
8,parking,1661
9,bar,1177


#### Class Exercise
Count the number of 'restaurants' inside each city of 'Metropolitan Manila'

In [6]:
#Code Here
pd.read_sql('''
SELECT g.name_2, count(p.amenity)
FROM gadm.ph as g
JOIN ph_point as p ON st_within(p.way,g.geom)
WHERE g.name_1 = 'Metropolitan Manila' AND p.amenity = 'restaurant'
GROUP BY 1
ORDER BY 2 DESC
''',conn)

,name_2,count
0,Quezon City,741
1,Makati City,678
2,Manila,263
3,Pasig City,218
4,Parañaque,166
5,Muntinlupa,161
6,Pasay City,150
7,San Juan,118
8,Marikina,102
9,Las Piñas,99


In [7]:
pd.read_sql('''
SELECT *
FROM gadm.ph as g
WHERE g.name_2 LIKE '%Navotas%'
''',conn)

,gid,gid_0,name_0,gid_1,name_1,nl_name_1,gid_2,name_2,varname_2,nl_name_2,type_2,engtype_2,cc_2,hasc_2,geom
0,969,PHL,Philippines,PHL.47_1,Metropolitan Manila,None,PHL.47.9_1,Navotas,None,None,Lungsod|Siyudad,City,137503,PH.MM.NV,0106000020E610000001000000010300000001000000C0...


#### Exercise
Create a table containing the number of each of the top 20 amenity types within each city/municipality in Metro Manila

| Province | City | restaurant | school | ... |

In [8]:
amenities = df3.amenity.to_list()

In [9]:
amenities

['restaurant',
 'school',
 'bank',
 'fast_food',
 'place_of_worship',
 'fuel',
 'pharmacy',
 'cafe',
 'parking',
 'bar',
 'townhall',
 'police',
 'atm',
 'community_centre',
 'bus_station',
 'clinic',
 'shelter',
 'toilets',
 'hospital',
 'post_office']

In [10]:
#Code Here
amenities = df3.amenity.to_list()
df_mm = pd.read_sql(f'''
SELECT g.name_1 as province, 
       g.name_2 as city,
       p.amenity as amenity,
       count(*) as counts
FROM gadm.ph as g
JOIN ph_point as p ON st_within(p.way,g.geom)
WHERE p.amenity IN {tuple(amenities)}
AND g.name_1 = 'Metropolitan Manila'
GROUP BY 1,2,3
ORDER BY 4 DESC
''',conn)
df_mm

,province,city,amenity,counts
0,Metropolitan Manila,Quezon City,restaurant,741
1,Metropolitan Manila,Makati City,restaurant,678
2,Metropolitan Manila,Quezon City,bank,471
3,Metropolitan Manila,Quezon City,fast_food,467
4,Metropolitan Manila,Makati City,bank,462
...,...,...,...,...
310,Metropolitan Manila,Pateros,post_office,1
311,Metropolitan Manila,Navotas,shelter,1
312,Metropolitan Manila,Pateros,hospital,1
313,Metropolitan Manila,Navotas,hospital,1


Now let's create one using the entire dataset:

In [11]:
#Code Here
amenities = df3.amenity.to_list()
df_data = pd.read_sql(f'''
SELECT g.name_1 as province, 
       g.name_2 as city,
       p.amenity as amenity,
       count(*) as counts
FROM gadm.ph as g
JOIN ph_point as p ON st_within(p.way,g.geom)
WHERE p.amenity IN {tuple(amenities)}
GROUP BY 1,2,3
ORDER BY 4 DESC
''',conn)
df_data

,province,city,amenity,counts
0,Metropolitan Manila,Quezon City,restaurant,741
1,Metropolitan Manila,Makati City,restaurant,678
2,Metropolitan Manila,Quezon City,bank,471
3,Metropolitan Manila,Quezon City,fast_food,467
4,Metropolitan Manila,Makati City,bank,462
...,...,...,...,...
8961,Quezon,Quezon,pharmacy,1
8962,Davao Oriental,Boston,townhall,1
8963,Capiz,Jamindan,hospital,1
8964,Maguindanao,Ampatuan,fuel,1


In [12]:
pivot_data = pd.pivot_table(df_data,values = 'counts', index = ['province','city'], columns = ['amenity'])
pivot_data.reset_index(inplace=True)
pivot_data.columns.name = None
pivot_data

,province,city,atm,bank,bar,bus_station,cafe,clinic,community_centre,fast_food,...,parking,pharmacy,place_of_worship,police,post_office,restaurant,school,shelter,toilets,townhall
0,Abra,Bangued,NaN,5.0,NaN,NaN,1.0,NaN,1.0,3.0,...,NaN,1.0,NaN,NaN,NaN,4.0,2.0,NaN,NaN,NaN
1,Abra,Manabo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
2,Abra,San Juan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
3,Abra,San Quintin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN
4,Agusan del Norte,Buenavista,1.0,NaN,NaN,NaN,NaN,NaN,1.0,3.0,...,NaN,2.0,3.0,NaN,NaN,NaN,7.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1334,Zamboanga del Sur,Margosatubig,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,NaN,NaN,3.0,NaN,NaN,1.0
1335,Zamboanga del Sur,Molave,1.0,1.0,NaN,1.0,1.0,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,1.0,1.0,NaN,NaN,NaN
1336,Zamboanga del Sur,Pagadian City,1.0,13.0,1.0,3.0,1.0,13.0,2.0,5.0,...,1.0,6.0,8.0,4.0,3.0,2.0,52.0,1.0,NaN,1.0
1337,Zamboanga del Sur,San Miguel,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2.0,NaN,NaN,8.0,NaN,NaN,6.0


### Import our Attribute Data

In [13]:
targets = pd.read_csv('financial_pop.csv')
targets

FileNotFoundError: [Errno 2] No such file or directory: 'financial_pop.csv'

In [ ]:
#Check the contents of the attribute data

In [ ]:
#Lets merge the population column to our data 
pop_df = pd.read_csv('financial_pop.csv', 
                     usecols=['shp_province', 'shp_municipality', 'pop'])
pop_df.columns = ['population', 'province', 'city']
merged_df = pd.merge(pivot_data, pop_df, on=['province', 'city'])
merged_df = merged_df.fillna(0)
merged_df

,province,city,atm,bank,bar,bus_station,cafe,clinic,community_centre,fast_food,...,pharmacy,place_of_worship,police,post_office,restaurant,school,shelter,toilets,townhall,population
0,Abra,Bangued,0.0,5.0,0.0,0.0,1.0,0.0,1.0,3.0,...,1.0,0.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,48163.0
1,Abra,Manabo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,10761.0
2,Abra,San Juan,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,9867.0
3,Abra,San Quintin,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,5438.0
4,Agusan del Norte,Buenavista,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,...,2.0,3.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,61614.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1323,Zamboanga del Sur,Margosatubig,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,3.0,0.0,0.0,1.0,37873.0
1324,Zamboanga del Sur,Molave,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,52006.0
1325,Zamboanga del Sur,Pagadian City,1.0,13.0,1.0,3.0,1.0,13.0,2.0,5.0,...,6.0,8.0,4.0,3.0,2.0,52.0,1.0,0.0,1.0,199060.0
1326,Zamboanga del Sur,San Miguel,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.0,0.0,0.0,8.0,0.0,0.0,6.0,19205.0


### Regression Model
We can now use our data in our machine learning model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

#### Class exercise
Use linear regression to check if we can predict the population using our amenity dataset

In [ ]:
# Prepare Features
X = merged_df.drop(['province','city','population'],axis  = 1)
X

,atm,bank,bar,bus_station,cafe,clinic,community_centre,fast_food,fuel,hospital,parking,pharmacy,place_of_worship,police,post_office,restaurant,school,shelter,toilets,townhall
0,0.0,5.0,0.0,0.0,1.0,0.0,1.0,3.0,4.0,0.0,0.0,1.0,0.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,2.0,0.0,0.0,2.0,3.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1323,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0,0.0,0.0,1.0
1324,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1325,1.0,13.0,1.0,3.0,1.0,13.0,2.0,5.0,12.0,4.0,1.0,6.0,8.0,4.0,3.0,2.0,52.0,1.0,0.0,1.0
1326,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,8.0,0.0,0.0,6.0


In [ ]:
# Prepare your Target
y = merged_df['population']
y

0        48163.0
1        10761.0
2         9867.0
3         5438.0
4        61614.0
          ...   
1323     37873.0
1324     52006.0
1325    199060.0
1326     19205.0
1327    861799.0
Name: population, Length: 1328, dtype: float64

In [ ]:
# Train Model
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 1)
lr = LinearRegression().fit(X_train, y_train)

In [ ]:
# Check Accuracy
print("training set score: %f" % lr.score(X_train, y_train))
print("test set score: %f" % lr.score(X_test, y_test))

training set score: 0.894820
test set score: 0.745669


### Exercise
Try to have the highest accuracy in predicting the population using our available datasets.

In [ ]:
df_filtered = merged_df[merged_df['population'] != 0]
df_filtered

,province,city,atm,bank,bar,bus_station,cafe,clinic,community_centre,fast_food,...,pharmacy,place_of_worship,police,post_office,restaurant,school,shelter,toilets,townhall,population
0,Abra,Bangued,0.0,5.0,0.0,0.0,1.0,0.0,1.0,3.0,...,1.0,0.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,48163.0
1,Abra,Manabo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,10761.0
2,Abra,San Juan,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,9867.0
3,Abra,San Quintin,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,5438.0
4,Agusan del Norte,Buenavista,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,...,2.0,3.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,61614.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1323,Zamboanga del Sur,Margosatubig,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,3.0,0.0,0.0,1.0,37873.0
1324,Zamboanga del Sur,Molave,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,52006.0
1325,Zamboanga del Sur,Pagadian City,1.0,13.0,1.0,3.0,1.0,13.0,2.0,5.0,...,6.0,8.0,4.0,3.0,2.0,52.0,1.0,0.0,1.0,199060.0
1326,Zamboanga del Sur,San Miguel,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.0,0.0,0.0,8.0,0.0,0.0,6.0,19205.0


In [ ]:
X2 = df_filtered.drop(['province','city','population'],axis  = 1)
X2

,atm,bank,bar,bus_station,cafe,clinic,community_centre,fast_food,fuel,hospital,parking,pharmacy,place_of_worship,police,post_office,restaurant,school,shelter,toilets,townhall
0,0.0,5.0,0.0,0.0,1.0,0.0,1.0,3.0,4.0,0.0,0.0,1.0,0.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,2.0,0.0,0.0,2.0,3.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1323,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0,0.0,0.0,1.0
1324,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1325,1.0,13.0,1.0,3.0,1.0,13.0,2.0,5.0,12.0,4.0,1.0,6.0,8.0,4.0,3.0,2.0,52.0,1.0,0.0,1.0
1326,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,8.0,0.0,0.0,6.0


In [ ]:
y2 = df_filtered['population']
y2

0        48163.0
1        10761.0
2         9867.0
3         5438.0
4        61614.0
          ...   
1323     37873.0
1324     52006.0
1325    199060.0
1326     19205.0
1327    861799.0
Name: population, Length: 1302, dtype: float64

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, random_state = 1)
lr2 = LinearRegression().fit(X2_train, y2_train)

In [ ]:
print("training set score: %f" % lr2.score(X2_train, y2_train))
print("test set score: %f" % lr2.score(X2_test, y2_test))

training set score: 0.889846
test set score: 0.843700
